# 🏦 Détection d'anomalies bancaires

Pipeline **non supervisé** qui identifie les transactions inhabituelles (candidates à investigation), avec une évaluation **honnête** (pas de label de vérité → pas de faux 99 %).

⚠️ Ce notebook détecte des **anomalies**, pas des fraudes prouvées.

In [ ]:
# ============ 1. IMPORTS ============
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============ 2. CHARGEMENT (UNE SEULE FOIS) ============
import os
DATA_PATH = '../data/bank_transactions_data_2.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'bank_transactions_data_2.csv'  # fallback : même dossier

df = pd.read_csv(DATA_PATH)
print(f"Données chargées : {len(df)} transactions, {df.shape[1]} colonnes")
df.head()

In [ ]:
# ============ 3. EXPLORATION ============
print(df.info())
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum())
print("\n--- Doublons ---")
print(df.duplicated().sum())

In [ ]:
# ============ 4. DATES + FEATURES TEMPORELLES ============
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Hour'] = df['TransactionDate'].dt.hour
df['DayOfWeek'] = df['TransactionDate'].dt.day_name()

# Ratio montant / solde (en %)
df['AmountToBalanceRatio'] = df['TransactionAmount'] / df['AccountBalance'] * 100
df[['TransactionDate', 'Hour', 'DayOfWeek', 'AmountToBalanceRatio']].head()

In [ ]:
# ============ 5. EDA — Distribution du ratio ============
plt.figure(figsize=(10, 5))
sns.histplot(df['AmountToBalanceRatio'], binwidth=5, color='teal')
plt.title('Distribution du ratio montant / solde (%)')
plt.xlabel('Ratio (%)')
plt.ylabel('Nombre de transactions')
plt.show()

tranche_0_5 = ((df['AmountToBalanceRatio'] >= 0) & (df['AmountToBalanceRatio'] < 5)).sum()
hors_100 = (df['AmountToBalanceRatio'] > 100).sum()
print(f"Transactions entre 0% et 5% : {tranche_0_5}")
print(f"Transactions > 100% (solde dépassé) : {hors_100}")

In [ ]:
# ============ 6. EDA — Volume par heure ============
plt.figure(figsize=(12, 5))
sns.countplot(x='Hour', data=df, palette='magma', order=range(24))
plt.title('Nombre de transactions par heure (0h - 23h)')
plt.xlabel('Heure')
plt.ylabel('Nombre de transactions')
plt.show()
print("→ Pic à 16h (fin de journée) = comportement légitime. Les heures creuses (01h-04h) sont à surveiller.")

In [ ]:
# ============ 7. EDA — Ratio médian par heure ============
ratio_par_heure = df.groupby('Hour')['AmountToBalanceRatio'].median().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(x='Hour', y='AmountToBalanceRatio', data=ratio_par_heure,
             marker='o', color='teal', linewidth=2.5)
plt.title('Ratio médian montant / solde au fil de la journée')
plt.xlabel('Heure (0h - 23h)')
plt.ylabel('Ratio médian (%)')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## EDA — Synthèse
- **16h** = pic de volume (fin de journée, comportement légitime)
- Les **heures creuses** (01h-04h) sont plus risquées pour les gros montants
- Une transaction qui **dépasse 100% du solde** est par nature anormale

→ Passons à la détection d'anomalies.

In [ ]:
# ============ 8. PIPELINE — Tri causal + Features ============
# Tri chronologique par compte (PAS par heure — évite la fuite du futur)
df = df.sort_values(['AccountID', 'TransactionDate']).reset_index(drop=True)

# Solde AVANT la transaction (AccountBalance = solde POST-transaction)
balance_avant = np.where(
    df['TransactionType'] == 'Debit',
    df['AccountBalance'] + df['TransactionAmount'],   # débit
    df['AccountBalance'] - df['TransactionAmount'],   # crédit
)
df['BalanceAvant'] = np.where(balance_avant > 0, balance_avant, df['AccountBalance'])
df['CorrectRatio'] = df['TransactionAmount'] / df['BalanceAvant'].replace(0, np.nan)

# Délai depuis la transaction précédente du même compte
df['TimeDeltaMinutes'] = df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds() / 60.0

# Baseline historique (moyenne des montants passés)
def past_mean(s):
    return s.shift(1).expanding(min_periods=3).mean()

df['HistoricalMeanAmount'] = df.groupby('AccountID')['TransactionAmount'].transform(past_mean)
df['HistoricalMeanAmount'] = df['HistoricalMeanAmount'].fillna(df['TransactionAmount'].mean())
df['AmountVsHistoryRatio'] = df['TransactionAmount'] / df['HistoricalMeanAmount']
print("Features créées : BalanceAvant, CorrectRatio, TimeDeltaMinutes, AmountVsHistoryRatio")

In [ ]:
# ============ 9. PIPELINE — Z-score robuste + Isolation Forest ============
# Modified Z-score sur log(ratio) — robuste à l'asymétrie
log_ratio = np.log1p(df['CorrectRatio'].fillna(0))
median_log = log_ratio.median()
mad_log = (log_ratio - median_log).abs().median() or 1e-6
df['ModifiedZScore'] = 0.6745 * (log_ratio - median_log) / mad_log

# Isolation Forest (multivarié)
features = ['TransactionAmount', 'CorrectRatio', 'TimeDeltaMinutes', 'LoginAttempts']
X = StandardScaler().fit_transform(df[features].fillna(0))
iso = IsolationForest(contamination=0.03, random_state=42)
df['IsoFlag'] = (iso.fit_predict(X) == -1).astype(int)
print("Z-score + Isolation Forest calculés")

In [ ]:
# ============ 10. SCORE DE RISQUE + SEUILS CALIBRÉS ============
# Seuil z-score auto-calibré (top 2% des plus extrêmes)
z_threshold = df['ModifiedZScore'].abs().quantile(0.98)

df['Rule_ZScore'] = (df['ModifiedZScore'].abs() > z_threshold).astype(int)
df['Rule_Ratio'] = (df['CorrectRatio'] > 10.0).astype(int)
df['Rule_Iso'] = df['IsoFlag']

# Score pondéré (poids documentés)
df['RiskScore'] = 2 * df['Rule_ZScore'] + 3 * df['Rule_Ratio'] + 2 * df['Rule_Iso']

In [ ]:
# ============ 11. ÉVALUATION HONNÊTE + EXPLICABILITÉ ============
def get_anomaly_reason(row):
    reasons = []
    if row['Rule_Ratio']: reasons.append('Ratio > 10x solde')
    if row['Rule_ZScore']: reasons.append('Z-score extrême')
    if row['Rule_Iso']: reasons.append('Anomalie multivariée (IsoForest)')
    return ' | '.join(reasons) if reasons else 'Normal'

df['AnomalyReason'] = df.apply(get_anomaly_reason, axis=1)
anomalies = df[df['RiskScore'] > 0].sort_values('RiskScore', ascending=False)
pct = len(anomalies) / len(df) * 100

print("=" * 55)
print("RAPPORT D'ÉVALUATION HONNÊTE")
print("=" * 55)
print(f"Volume analysé        : {len(df)} transactions")
print(f"Anomalies signalées   : {len(anomalies)} ({pct:.1f} %)")
print(f"  - Ratio > 10x       : {df['Rule_Ratio'].sum()}")
print(f"  - Z-score extrême   : {df['Rule_ZScore'].sum()}")
print(f"  - Isolation Forest  : {df['Rule_Iso'].sum()}")
print()
print("Top 10 des candidats à investigation :")
cols = ['TransactionID','AccountID','TransactionType','TransactionAmount',
        'BalanceAvant','CorrectRatio','RiskScore','AnomalyReason']
anomalies[cols].head(10)

In [ ]:
# ============ 12. EXPORTS ============
cols_export = ['TransactionID','AccountID','TransactionDate','TransactionType',
               'TransactionAmount','BalanceAvant','CorrectRatio','RiskScore','AnomalyReason']
anomalies[cols_export].to_csv('../output/investigation_prioritaire.csv', index=False)
df.to_csv('../output/analyse_globale.csv', index=False)
print("✅ Exports : output/investigation_prioritaire.csv + output/analyse_globale.csv")